In [3]:
"""
07_kmeans_clustering.py -- ScoutAI K-Means Player Segmentation

Performs unsupervised machine learning (K-Means) to cluster players into 
distinct profiles based on age, appearances, caps, and market value. 
Uses PCA for 2D visualization and automatically saves outputs to their 
respective directories.
"""

import os
import sys
import pandas as pd
import numpy as np
import sqlalchemy
import matplotlib
matplotlib.use('Agg')  # Headless-safe
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from dotenv import load_dotenv, find_dotenv

# ==========================================
# 0. CONFIG & SETUP
# ==========================================

# Load environment variables
load_dotenv(find_dotenv())

IMAGES_DIR = "images"
DATA_DIR = "data"

# Ensure output directories exist
for directory in [IMAGES_DIR, DATA_DIR]:
    os.makedirs(directory, exist_ok=True)

def main():
    # ==========================================
    # 1. DATABASE CONNECTION & DATA PREP
    # ==========================================
    db_url = os.getenv('DB_URL')
    if not db_url:
        print("[ERROR] DB_URL not found. Please ensure your .env file exists and is configured correctly.")
        sys.exit(1)

    print("[SYSTEM] Fetching data for Clustering...")
    try:
        engine = sqlalchemy.create_engine(db_url)
        df = pd.read_sql("SELECT * FROM view_scout_master", engine)
    except Exception as e:
        print(f"[ERROR] Database connection failed: {e}")
        sys.exit(1)

    df['current_market_value'] = pd.to_numeric(df['current_market_value'], errors='coerce').fillna(0)
    df = df[df['current_market_value'] > 0].copy()

    # Use log_market_value to prevent extreme outliers from skewing clusters
    cluster_features = ['age', 'total_appearances', 'international_caps', 'log_market_value']

    for col in cluster_features:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    X_cluster = df[cluster_features].copy()

    # ==========================================
    # 2. STANDARDIZATION (Z-Score)
    # ==========================================
    print("[SYSTEM] Standardizing features...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_cluster)

    # ==========================================
    # 3. K-MEANS ALGORITHM
    # ==========================================
    print("[SYSTEM] Running K-Means Algorithm (k=4)...")
    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    df['Cluster'] = kmeans.fit_predict(X_scaled)

    # ==========================================
    # 4. DIMENSIONALITY REDUCTION (PCA)
    # ==========================================
    pca = PCA(n_components=2)
    pca_components = pca.fit_transform(X_scaled)
    df['PCA1'] = pca_components[:, 0]
    df['PCA2'] = pca_components[:, 1]
    
    print(f"[SYSTEM] PCA explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, "
          f"PC2={pca.explained_variance_ratio_[1]:.1%}, "
          f"total={sum(pca.explained_variance_ratio_):.1%}")

    # ==========================================
    # 5. CLUSTER PROFILING
    # ==========================================
    print("[SYSTEM] Profiling and labeling clusters...")
    cluster_centers = scaler.inverse_transform(kmeans.cluster_centers_)
    cluster_df = pd.DataFrame(cluster_centers, columns=cluster_features)
    cluster_df['avg_market_value'] = np.expm1(cluster_df['log_market_value'])

    def label_cluster(row):
        if row['avg_market_value'] > 40_000_000:
            return "Elite Superstars"
        elif row['age'] > 29:
            return "Experienced Veterans"
        elif row['age'] < 24 and row['total_appearances'] < 100:
            return "Developing Prospects"
        else:
            return "Prime / Core Squad"

    cluster_df['Cluster_Name'] = cluster_df.apply(label_cluster, axis=1)

    # Disambiguate duplicate labels if any
    name_counts = cluster_df['Cluster_Name'].value_counts()
    duplicate_names = name_counts[name_counts > 1].index
    if len(duplicate_names) > 0:
        print(f"[WARNING] Duplicate labels detected: {list(duplicate_names)} -- disambiguating.")
        seen = {}
        new_names = []
        for name in cluster_df['Cluster_Name']:
            if name in duplicate_names:
                seen[name] = seen.get(name, 0) + 1
                new_names.append(f"{name} ({seen[name]})")
            else:
                new_names.append(name)
        cluster_df['Cluster_Name'] = new_names

    cluster_mapping = cluster_df['Cluster_Name'].to_dict()
    df['Player_Profile'] = df['Cluster'].map(cluster_mapping)

    # ==========================================
    # 6. VISUALIZATION
    # ==========================================
    print("[SYSTEM] Generating Segmentation Scatter Plot...")
    plt.figure(figsize=(14, 8))
    sns.scatterplot(
        x='PCA1', y='PCA2',
        hue='Player_Profile',
        palette='tab10',
        data=df,
        alpha=0.6,
        edgecolor='k',
        s=60,
    )
    plt.title("ScoutAI: Player Segmentation (K-Means Clustering)", fontsize=16, fontweight='bold')
    plt.xlabel("Experience & Value Dimension (PCA1)", fontsize=12)
    plt.ylabel("Age & International Caps Dimension (PCA2)", fontsize=12)
    plt.legend(title='Machine Learned Profiles', fontsize=10, title_fontsize=12, loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    plot_path = os.path.join(IMAGES_DIR, 'kmeans_player_segmentation.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[SUCCESS] Segmentation plot saved to '{plot_path}'")

    # ==========================================
    # 7. SUMMARY REPORT & EXPORT
    # ==========================================
    report_lines = []
    
    header = (
        "\n==========================================\n"
        " 🧬 K-MEANS CLUSTER CENTERS (AVERAGES)\n"
        "==========================================\n"
    )
    print(header, end="")
    report_lines.append(header)

    for idx, row in cluster_df.iterrows():
        n_players = (df['Cluster'] == idx).sum()
        profile_block = (
            f" Profile: {row['Cluster_Name']}  (n={n_players})\n"
            f"  * Avg Age: {row['age']:.1f}\n"
            f"  * Avg Appearances: {row['total_appearances']:.0f}\n"
            f"  * Avg Int. Caps: {row['international_caps']:.1f}\n"
            f"  * Avg Value: €{row['avg_market_value']:,.0f}\n\n"
        )
        print(profile_block, end="")
        report_lines.append(profile_block)
        
    footer = "==========================================\n"
    print(footer, end="")
    report_lines.append(footer)

    # Save numeric summary to TXT in DATA_DIR
    txt_export_path = os.path.join(DATA_DIR, 'kmeans_cluster_summary.txt')
    with open(txt_export_path, 'w', encoding='utf-8') as f:
        f.writelines(report_lines)
    
    print(f"[SUCCESS] Cluster summary report safely exported to '{txt_export_path}'")

if __name__ == "__main__":
    main()

[SYSTEM] Fetching data for Clustering...
[SYSTEM] Standardizing features...
[SYSTEM] Running K-Means Algorithm (k=4)...
[SYSTEM] PCA explained variance: PC1=49.2%, PC2=26.8%, total=76.1%
[SYSTEM] Profiling and labeling clusters...
[WARNING] Duplicate labels detected: ['Experienced Veterans'] -- disambiguating.
[SYSTEM] Generating Segmentation Scatter Plot...
[SUCCESS] Segmentation plot saved to 'images\kmeans_player_segmentation.png'

 🧬 K-MEANS CLUSTER CENTERS (AVERAGES)
 Profile: Prime / Core Squad  (n=5256)
  * Avg Age: 24.7
  * Avg Appearances: 64
  * Avg Int. Caps: 7.2
  * Avg Value: €4,094,070

 Profile: Experienced Veterans (1)  (n=1193)
  * Avg Age: 31.6
  * Avg Appearances: 278
  * Avg Int. Caps: 56.3
  * Avg Value: €2,980,926

 Profile: Experienced Veterans (2)  (n=5318)
  * Avg Age: 32.1
  * Avg Appearances: 82
  * Avg Int. Caps: 4.1
  * Avg Value: €356,233

 Profile: Developing Prospects  (n=8613)
  * Avg Age: 23.0
  * Avg Appearances: 7
  * Avg Int. Caps: 2.3
  * Avg Value